In [ ]:
# =============================================================================
# BOOTSTRAP: instalacion automatica de dependencias (self-contained, Linux-ready)
# =============================================================================
import importlib
import subprocess
import sys


def _bootstrap_dependencias(paquetes):
    faltantes = []
    for modulo, paquete_pip in paquetes.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltantes.append(paquete_pip)
    if faltantes:
        print(f"[INFO] Instalando dependencias faltantes: {', '.join(faltantes)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])


_bootstrap_dependencias({
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlalchemy": "sqlalchemy",
    "pyodbc": "pyodbc",
})

# hasattr(...): bajo Jupyter/ipykernel, sys.stdout/stderr son un OutStream
# propio que NO tiene .reconfigure() (solo lo tiene un TextIOWrapper real,
# el caso de un script corrido por CPython en terminal/cron/systemd) -- sin
# este chequeo, la celda revienta con AttributeError apenas se corre como
# notebook en un kernel.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import os
import hashlib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, bindparam
from urllib.parse import quote_plus
import datetime


In [ ]:
# =============================================================================
# CONFIGURACION (autocontenido: sin dependencias de otro archivo)
# =============================================================================

def _verificar_driver_odbc(nombre_driver="ODBC Driver 17 for SQL Server"):
    """
    El driver ODBC de SQL Server es un paquete de sistema operativo, no
    instalable por pip. Se valida su presencia antes de conectar para fallar
    con un mensaje claro en vez de un traceback criptico de pyodbc.
    """
    import pyodbc
    disponibles = pyodbc.drivers()
    if nombre_driver not in disponibles:
        raise RuntimeError(
            f"No se encontro el driver ODBC '{nombre_driver}' en este servidor.\n"
            f"Drivers disponibles: {disponibles or '(ninguno)'}\n"
            "Instalar en Debian/Ubuntu:\n"
            "  curl -sSL https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -\n"
            "  curl -sSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list\n"
            "  sudo apt-get update && ACCEPT_EULA=Y sudo apt-get install -y msodbcsql17 unixodbc-dev"
        )


_verificar_driver_odbc()

conn_str_work = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27,1433;"
    "DATABASE=SANDBOX_BRONZE;"
    "UID=sa;"
    "PWD=M4rfr1g.2023*;"
    "TrustServerCertificate=yes;"
)
# fast_executemany=True: pyodbc manda cada lote de INSERT en un solo viaje al
# servidor en vez de una ida y vuelta por fila.
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_work}", fast_executemany=True)

conn_str_mds = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27\\mfgbi;"
    "DATABASE=MDS_PRODU;"
    "UID=pbi_report;"
    "PWD=Marfrig2024;"
    "TrustServerCertificate=yes;"
)
engine_mds = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_mds}")


In [ ]:
# =============================================================================
# TABLA DE DINAMICAS
# -----------------------------------------------------------------------------
# Multiplicador aplicado a Precio_BASE cuando la fila trae una dinamica de
# promocion de TEXTO ('2x1', '2do al 50%', ...) y no hay Precio OFERTA.
#
# Las dinamicas NUMERICAS (fraccion de descuento: 0.25 -> paga el 75%) NO
# figuran aca: se resuelven en calcular_precio_efectivo parseando el numero
# (acepta coma o punto decimal), en vez de por un diccionario de strings
# exactos. Antes esta tabla tambien listaba las fracciones numericas como
# claves de texto con coma decimal ('0,25': 0.75, con un parche adicional
# '0,55000000000000004': 0.45 para un caso de redondeo de float) -- ese
# lookup fallaba en cuanto la fuente entregaba el mismo numero con otro
# separador decimal o con distinta precision, que es justamente el bug
# reportado el 13/07/2026: Dinamica=0.25 no aplicaba ningun descuento porque
# el diccionario solo tenia la clave '0,25' (coma), no '0.25' (punto).
# =============================================================================

TABLA_DINAMICAS = {
    '2do al 10%': 0.95, '2do al 15%': 0.925, '2do al 20%': 0.90,
    '2do al 25%': 0.875, '2do al 30%': 0.85, '2do al 35%': 0.825,
    '2do al 40%': 0.80, '2do al 45%': 0.775, '2do al 50%': 0.75,
    '2do al 55%': 0.725, '2do al 60%': 0.70, '2do al 65%': 0.675,
    '2do al 70%': 0.65, '2do al 75%': 0.625, '2do al 80%': 0.60,
    '2do al 85%': 0.575, '2do al 90%': 0.55, '2do al 95%': 0.525,
    '2x1': 0.50, '3x1': 0.333333333, '3x2': 0.666666667,
    '4x2': 0.50, '4x3': 0.75, '5x3': 0.60, '5x4': 0.80,
    'Jumbo al 50%': 1.0, 'Jumbo al 70%': 1.0, 'Jumbo al 100%': 1.0,
}


In [ ]:
# =============================================================================
# DETECCION DE OUTLIERS - PARAMETROS
# -----------------------------------------------------------------------------
# Formula asimetrica basada en p50 + IQR (acordada con analista, 23/04/2026),
# corregida el 13-15/07/2026 a pedido de Pricing (mail "BI Horus -
# Visualizacion y tablas"):
#   Limite superior: max(p50 + 8xIQR,  p50x2.5)
#   Limite inferior: min(p50 - 8xIQR,  p50x0.25)
#
# p50x[0.25, 2.5] actua como PISO de banda minima, nunca como techo: antes
# se tomaba min/max al reves (min para el superior, max para el inferior),
# es decir el rango 0.25-2.5x RECORTABA una banda que 8xIQR ya habia hecho
# mas ancha. En grupos con IQR chico (precio muy uniforme entre PDVs) esto
# marcaba como outliers precios legitimos apenas mas dispersos que el resto
# -- el caso reportado: EAN 1000003 (Pollo fresco xkg), Precio_BASE=8500,
# limite superior viejo=7771 (=p50x2.5, recortando una banda que ya era mas
# ancha por IQR). Con el min/max corregido, el limite usa la banda mas ancha
# en vez de la mas angosta.
#
# La deteccion se evalua sobre Precio_BASE, no sobre Precio_Dinamizado: el
# precio dinamizado trae descuentos promocionales legitimos que caen fuera
# de la distribucion normal del grupo y antes se marcaban como falsos
# outliers -- el caso reportado: EAN 7790070037967, Precio_BASE=3159 (dentro
# de [2100, 5300]) pero Precio_Dinamizado=1719 (fuera), marcado outlier por
# evaluar la columna equivocada.
#
# La deteccion agrupa por EAN_Final + Fecha (EAN canonico post-reemplazo).
# IMPORTANTE: se usa EAN_Final y no EAN para que productos con multiples EANs
# de entrada (reemplazos) formen UNA SOLA distribucion. Agrupar por EAN
# original genera distribuciones distintas para el mismo producto, produciendo
# limites inconsistentes al consolidarlos por EAN_Final en capas downstream.
#
# Requiere minimo 4 registros por grupo.
# =============================================================================


In [ ]:
# =============================================================================
# FUNCIONES DE TRANSFORMACION
# =============================================================================

def normalizar_columnas_formato_viejo(df):
    """
    Mapea columnas del formato viejo (Base Reporte / Base Status Nacional / Base)
    al esquema unificado. No tiene Dinamica ni Precio OFERTA, por lo que
    Precio_Dinamizado = Precio BASE directamente.
    """
    mapeo = {
        'Descripción Boca': 'Descripcion_Boca',
        'Cód Boca Marfrig': 'Cod_Boca',
        'Cód Cli Marfrig': 'Cod_Cliente',
        'Descripción Cliente': 'Descripcion_Cliente',
        'Canal IC': 'Canal',
        'Descripción SKU': 'Descripcion_SKU',
        'Familia': 'Familia',
        'Empresa': 'Empresa',
        'Marca': 'Marca',
        'Cód SKU Marfrig': 'Cod_SKU',
        'Fecha': 'Fecha',
        'Precio': 'Precio_BASE',
        '_archivo_origen': '_archivo_origen',
        '_hoja_origen': '_hoja_origen',
    }
    mapeo_existente = {k: v for k, v in mapeo.items() if k in df.columns}
    df = df.rename(columns=mapeo_existente)

    for col in ['Dinamica', 'Precio_POR_MAYOR', 'Precio_OFERTA', 'Supervisor', 'EAN']:
        if col not in df.columns:
            df[col] = np.nan

    df['_formato'] = 'viejo'
    return df


def normalizar_columnas_formato_nuevo(df):
    """
    Mapea columnas del formato nuevo (Planilla nueva) al esquema unificado.
    Contiene Precio BASE, Dinamica, Precio POR MAYOR y Precio OFERTA.
    """
    mapeo = {
        'Supervisor': 'Supervisor',
        'Descripción Boca': 'Descripcion_Boca',
        'Cod boca': 'Cod_Boca',
        'EAN': 'EAN',
        'Familia': 'Familia',
        'Descripción SKU': 'Descripcion_SKU',
        'Precio BASE': 'Precio_BASE',
        'Dinámica': 'Dinamica',
        'Precio POR MAYOR': 'Precio_POR_MAYOR',
        'Precio OFERTA': 'Precio_OFERTA',
        'Día': 'Dia',
        'Mes': 'Mes',
        '_archivo_origen': '_archivo_origen',
        '_hoja_origen': '_hoja_origen',
    }
    mapeo_existente = {k: v for k, v in mapeo.items() if k in df.columns}
    df = df.rename(columns=mapeo_existente)
    df['_formato'] = 'nuevo'
    return df


def construir_fecha(df):
    """
    Construye columna Fecha unificada:
    - Formato viejo: columna Fecha directa.
    - Formato nuevo: construida desde columnas Dia y Mes.
      El anio se extrae del nombre del archivo (_archivo_origen) buscando
      el primer numero de 4 digitos en el nombre (ej: 'SNP_2025_Semana10.xlsx' -> 2025).
    """
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
    else:
        df['_anio'] = df['_archivo_origen'].str.extract(r'(\d{4})').astype(float)
        df['Fecha'] = pd.to_datetime(dict(
            year=df['_anio'],
            month=df['Mes'],
            day=df['Dia']
        ), errors='coerce')
        df = df.drop(columns=['_anio'])
    return df


def normalizar_fecha_lunes(df):
    """
    Normaliza la fecha relevada al lunes de su semana.
    Garantiza comparabilidad con Scanntech (que tambien normaliza a lunes).
    weekday(): 0=Lunes, 6=Domingo. Restando weekday() dias se obtiene el lunes.
    """
    df['Fecha'] = df['Fecha'] - pd.to_timedelta(df['Fecha'].dt.weekday, unit='D')
    return df


def calcular_precio_efectivo(df):
    """
    Regla de negocio para Precio_Dinamizado (comparabilidad con COPI):
    1. Si existe Precio OFERTA -> usar como precio final.
    2. Si no, si existe Dinamica con multiplicador -> Precio BASE x Multiplicador.
    3. Si no -> usar Precio BASE.

    La Dinamica puede venir como fraccion de descuento numerica ('0,25',
    '0.25' o el float 0.25 segun el formato de la celda de origen) ->
    multiplicador (1 - fraccion); o como texto promocional ('2x1',
    '2do al 50%') -> TABLA_DINAMICAS.

    Las fracciones numericas se resuelven parseando el valor (aceptando coma
    o punto decimal) en vez de buscar el string exacto en un diccionario: un
    lookup por string exacto se rompe apenas la fuente entrega el mismo
    numero con otro separador decimal, que es justamente lo que causaba que
    Dinamica=0.25 no aplicara ningun descuento (bug reportado 13/07/2026).

    Las dinamicas que no se resuelven por ninguna via caen al caso 3 (precio
    base sin descuento) y se REPORTAN: un precio promocional tratado como
    base infla el indice sin dejar rastro si esto pasa en silencio.
    """
    df['Precio_BASE']      = pd.to_numeric(df['Precio_BASE'],      errors='coerce')
    df['Precio_OFERTA']    = pd.to_numeric(df['Precio_OFERTA'],    errors='coerce')
    df['Precio_POR_MAYOR'] = pd.to_numeric(df['Precio_POR_MAYOR'], errors='coerce')

    dinamica_str = df['Dinamica'].astype(str).str.strip()

    # 1) Codigos promocionales de texto ('2x1', '2do al 50%', ...)
    multiplicador = dinamica_str.map(TABLA_DINAMICAS)

    # 2) Fraccion de descuento numerica, con coma o punto decimal
    dinamica_num = pd.to_numeric(dinamica_str.str.replace(',', '.', regex=False), errors='coerce')
    es_fraccion = multiplicador.isna() & (dinamica_num > 0) & (dinamica_num < 1)
    multiplicador = multiplicador.where(~es_fraccion, 1 - dinamica_num)

    # Trazabilidad: dinamicas que no se pudieron resolver (no se descartan en silencio)
    sin_mapear = dinamica_str[
        dinamica_str.notna()
        & ~dinamica_str.isin(['', 'nan', 'None', '<NA>'])
        & multiplicador.isna()
    ]
    if len(sin_mapear) > 0:
        detalle = sin_mapear.value_counts().head(20)
        print(f"[WARN] {len(sin_mapear):,} filas con Dinamica NO reconocida "
              f"(caen a Precio_BASE sin descuento). Agregar a TABLA_DINAMICAS:")
        print(detalle.to_string())

    precio_con_dinamica = np.where(
        multiplicador.notna(),
        df['Precio_BASE'] * multiplicador,
        np.nan
    )

    df['Precio_Dinamizado'] = np.where(
        df['Precio_OFERTA'].notna() & (df['Precio_OFERTA'] > 0),
        df['Precio_OFERTA'],
        np.where(
            pd.Series(precio_con_dinamica, index=df.index).notna(),
            precio_con_dinamica,
            df['Precio_BASE']
        )
    )

    return df


def calcular_limites_outlier(serie):
    """
    Calcula limites de outlier con formula asimetrica basada en p50 + IQR.
    Requiere al menos 4 observaciones para ser estadisticamente valido.

    p50 x [0.25, 2.5] es el PISO de banda minima (ver bloque de parametros
    mas arriba para el detalle del bug corregido):
        Superior: max(p50 + 8xIQR, p50x2.5)
        Inferior: min(p50 - 8xIQR, p50x0.25)

    Retorna: (lim_inf, lim_sup, p50, iqr)
    Los valores p50 e iqr se guardan en SNP_outliers para facilitar el diagnostico.
    """
    q1  = serie.quantile(0.25)
    q3  = serie.quantile(0.75)
    p50 = serie.quantile(0.50)
    iqr = q3 - q1
    lim_sup = max(p50 + 8 * iqr, p50 * 2.5)
    lim_inf = min(p50 - 8 * iqr, p50 * 0.25)
    return lim_inf, lim_sup, p50, iqr


def detectar_outliers(df):
    """
    Deteccion de outliers por EAN_Final/Fecha con formula asimetrica p50 + IQR,
    evaluada sobre Precio_BASE (ver bloque de parametros mas arriba para el
    detalle de los dos bugs corregidos).

    Columnas de salida:
        _es_outlier    : True si el precio base es outlier o no tiene precio valido.
        _motivo_outlier: descripcion del motivo.
        _lim_inf       : limite inferior del grupo EAN_Final/Fecha.
        _lim_sup       : limite superior del grupo EAN_Final/Fecha.
        _p50           : mediana del grupo (para diagnostico).
        _iqr           : rango intercuartilico del grupo (para diagnostico).
    """
    precios    = df['Precio_BASE']
    sin_precio = precios.isna()

    # Agrupar por EAN_Final (canonico) + Fecha - minimo 4 registros por grupo
    limites = {
        k: calcular_limites_outlier(g['Precio_BASE'].dropna())
        for k, g in df.groupby(['EAN_Final', 'Fecha'])
        if len(g['Precio_BASE'].dropna()) >= 4
    }

    keys = list(zip(df['EAN_Final'], df['Fecha']))
    df['_lim_inf'] = [limites[k][0] if k in limites else None for k in keys]
    df['_lim_sup'] = [limites[k][1] if k in limites else None for k in keys]
    df['_p50']     = [limites[k][2] if k in limites else None for k in keys]
    df['_iqr']     = [limites[k][3] if k in limites else None for k in keys]

    tiene_limites = df['_lim_inf'].notna()

    df['_es_outlier'] = (
        sin_precio |
        (tiene_limites & (precios < df['_lim_inf'])) |
        (tiene_limites & (precios > df['_lim_sup']))
    )

    df['_motivo_outlier'] = np.where(
        sin_precio, 'Sin precio',
        np.where(df['_es_outlier'], 'Outlier por EAN/Fecha', None)
    )

    return df


In [ ]:
# =============================================================================
# COLUMNAS FINALES
# =============================================================================

COLUMNAS_SILVER = [
    'Fecha', 'Codigo_PDV', 'Descripcion_Boca', 'Cadena', 'Cod_Boca',
    'Cod_Cliente', 'Descripcion_Cliente',
    'EAN', 'EAN_Final',
    'Familia', 'Empresa', 'Marca', 'Segmento',
    'Cod_SKU', 'Descripcion_SKU', 'Marca_Propia',
    'Precio_BASE', 'Dinamica', 'Precio_POR_MAYOR', 'Precio_OFERTA',
    'Precio_Dinamizado',
    'Supervisor', 'Canal',
    '_formato', '_archivo_origen', '_hoja_origen'
]

# _p50 e _iqr se agregan para permitir diagnostico de outliers sin tener que
# recalcular desde los datos originales.
COLUMNAS_OUTLIERS = COLUMNAS_SILVER + [
    '_es_outlier', '_motivo_outlier', '_lim_inf', '_lim_sup', '_p50', '_iqr'
]


In [ ]:
# =============================================================================
# CARGA INCREMENTAL POR SEMANA
# -----------------------------------------------------------------------------
# Antes: cada corrida leia TODO PRC.SNP_raw, recalculaba todo y reemplazaba
# SNP_silver y SNP_outliers enteras (DROP + CREATE + INSERT en chunks de 1000).
#
# Ahora la unidad de trabajo es la SEMANA (Fecha = lunes normalizado):
#   1. Nada en silver cruza semanas: los limites de outlier se calculan por
#      EAN_Final + Fecha y el resto de las transformaciones son por fila.
#   2. bronze_snp registra cada Excel cargado en PRC.SNP_processed_files, y
#      cada fila de SNP_raw lleva _archivo_origen. Silver guarda en
#      PRC.SNP_silver_archivos que version (fecha_proceso de bronze) de cada
#      archivo proceso y en que lunes cayeron sus filas.
#   3. Un archivo nuevo puede aportar filas a un lunes que ya tenia filas de
#      otro archivo (y los limites de outlier de ese lunes cambian). Por eso,
#      a partir de los lunes tocados por los archivos pendientes, se releen
#      TODOS los archivos que ya aportaban a esos lunes y se recalculan los
#      lunes completos.
#   4. DELETE de los lunes afectados + INSERT + control en UNA transaccion.
#   5. PRC.SNP_silver_semanas (Fecha -> fecha_proceso) es la senal que consume
#      gold_scanntech_snp.
#
# Full rebuild (todo de nuevo): HORUS_FULL_REBUILD=1, cambio en el maestro de
# EANs de MDS (afecta a toda la historia), cambio de VERSION_LOGICA, primera
# corrida o tablas inexistentes.
# HORUS_DRY_RUN=1: muestra el plan sin escribir nada.
#
# Filas sin fecha valida (Dia/Mes/anio no parseables) no pertenecen a ninguna
# semana y no pueden mantenerse de forma incremental: se descartan con [WARN]
# indicando archivo y cantidad. Antes llegaban a silver con Fecha NULL, donde
# no aportaban a ningun indice semanal.
# =============================================================================

SCHEMA           = 'PRC'
TABLA_BRONZE     = 'SNP_raw'
TABLA_ARCHIVOS_BZ = 'SNP_processed_files'   # la mantiene bronze_snp
TABLA_SILVER     = 'SNP_silver'
TABLA_OUTLIERS   = 'SNP_outliers'
TABLA_ARCHIVOS   = 'SNP_silver_archivos'    # archivo -> version procesada + lunes que toca
TABLA_SEMANAS    = 'SNP_silver_semanas'     # Fecha (lunes) -> fecha_proceso (la consume gold)
TABLA_CONTROL    = 'horus_control'
CLAVE_MAESTROS   = 'silver_snp.maestros'
VERSION_LOGICA   = '2026-11-17'             # subir al cambiar reglas -> fuerza full rebuild
CHUNK_SIZE       = 20_000


def _flag_env(nombre):
    return os.environ.get(nombre, '').strip().lower() in ('1', 'true', 'yes', 'si')


FULL_REBUILD_ENV = _flag_env('HORUS_FULL_REBUILD')
DRY_RUN          = _flag_env('HORUS_DRY_RUN')


def fingerprint_maestros(*dfs):
    h = hashlib.sha256(VERSION_LOGICA.encode())
    for df in dfs:
        d = df.astype(str)
        d = d.sort_values(list(d.columns)).reset_index(drop=True)
        h.update(pd.util.hash_pandas_object(d, index=False).values.tobytes())
    return h.hexdigest()


def _in_clause(sql, nombre):
    return text(sql).bindparams(bindparam(nombre, expanding=True))


def tabla_existe(conn, tabla):
    return conn.execute(text("SELECT OBJECT_ID(:n, 'U')"), {'n': f'{SCHEMA}.{tabla}'}).scalar() is not None


def asegurar_tablas_control(conn):
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_ARCHIVOS}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_ARCHIVOS}] (
            nombre_archivo       NVARCHAR(450) PRIMARY KEY,
            fecha_proceso_bronze DATETIME NULL,
            semanas              NVARCHAR(MAX) NULL,
            fecha_proceso        DATETIME DEFAULT GETDATE()
        )
    """))
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_SEMANAS}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_SEMANAS}] (
            Fecha          DATE PRIMARY KEY,
            filas          INT,
            filas_outliers INT,
            fecha_proceso  DATETIME DEFAULT GETDATE()
        )
    """))
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_CONTROL}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_CONTROL}] (
            clave NVARCHAR(200) PRIMARY KEY,
            valor NVARCHAR(MAX),
            fecha DATETIME DEFAULT GETDATE()
        )
    """))


def leer_control(conn, clave):
    return conn.execute(text(f"SELECT valor FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"),
                        {'k': clave}).scalar()


def escribir_control(conn, clave, valor):
    conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"), {'k': clave})
    conn.execute(text(f"INSERT INTO [{SCHEMA}].[{TABLA_CONTROL}] (clave, valor) VALUES (:k, :v)"),
                 {'k': clave, 'v': valor})


def leer_archivos_bronze(conn):
    """{nombre_archivo: fecha_proceso} segun bronze_snp (ultima carga de cada archivo)."""
    if not tabla_existe(conn, TABLA_ARCHIVOS_BZ):
        return {}
    rows = conn.execute(text(
        f"SELECT nombre_archivo, MAX(fecha_proceso) FROM [{SCHEMA}].[{TABLA_ARCHIVOS_BZ}] GROUP BY nombre_archivo"
    )).fetchall()
    return {r[0]: r[1] for r in rows}


def _fechas_a_texto(fechas):
    return ','.join(f.isoformat() for f in sorted(fechas))


def _texto_a_fechas(s):
    return {datetime.date.fromisoformat(x) for x in (s or '').split(',') if x}


def leer_archivos_silver(conn):
    """{nombre_archivo: (fecha_proceso_bronze, {lunes})} procesado por silver."""
    rows = conn.execute(text(
        f"SELECT nombre_archivo, fecha_proceso_bronze, semanas FROM [{SCHEMA}].[{TABLA_ARCHIVOS}]"
    )).fetchall()
    return {r[0]: (r[1], _texto_a_fechas(r[2])) for r in rows}


def determinar_pendientes(archivos_bz, archivos_sv):
    """
    cambiados : archivos nuevos en bronze o recargados (fecha_proceso distinta).
    eliminados: archivos que silver proceso y ya no figuran en bronze.
    """
    cambiados, motivos = [], {}
    for nombre in sorted(archivos_bz):
        fp = archivos_bz[nombre]
        prev = archivos_sv.get(nombre)
        if prev is None:
            motivo = 'nuevo en bronze'
        elif prev[0] != fp:
            motivo = f'recargado en bronze ({prev[0]} -> {fp})'
        else:
            continue
        cambiados.append(nombre)
        motivos[nombre] = motivo
    eliminados = sorted(n for n in archivos_sv if n not in archivos_bz)
    return cambiados, eliminados, motivos


def leer_bronze(archivos=None):
    """Lee SNP_raw completo (archivos=None) o solo las filas de los archivos indicados."""
    if archivos is None:
        chunks = pd.read_sql(f"SELECT * FROM [{SCHEMA}].[{TABLA_BRONZE}]", engine, chunksize=100_000)
        return pd.concat(chunks, ignore_index=True)
    if not archivos:
        return pd.read_sql(f"SELECT TOP 0 * FROM [{SCHEMA}].[{TABLA_BRONZE}]", engine)
    partes = []
    with engine.connect() as conn:
        for i in range(0, len(archivos), 500):
            stmt = _in_clause(
                f"SELECT * FROM [{SCHEMA}].[{TABLA_BRONZE}] WHERE _archivo_origen IN :archivos", 'archivos')
            partes.append(pd.read_sql(stmt, conn, params={'archivos': list(archivos[i:i + 500])}))
    return pd.concat(partes, ignore_index=True)


def _a_datetime(fechas):
    return [datetime.datetime.combine(f, datetime.time()) for f in fechas]


def borrar_valores(conn, tabla, columna, valores):
    if not valores:
        return
    for i in range(0, len(valores), 500):
        stmt = _in_clause(f"DELETE FROM [{SCHEMA}].[{tabla}] WHERE {columna} IN :vals", 'vals')
        conn.execute(stmt, {'vals': valores[i:i + 500]})


def crear_tabla_desde_df(conn, df, tabla):
    """
    Crea (o recrea) la tabla con el esquema inferido del DataFrame COMPLETO y
    un indice clustered por Fecha (acelera el DELETE por semana del incremental).
    """
    df.head(0).to_sql(tabla, con=conn, schema=SCHEMA, if_exists='replace', index=False)
    conn.execute(text(f"CREATE CLUSTERED INDEX [IX_{tabla}_Fecha] ON [{SCHEMA}].[{tabla}] (Fecha)"))


def insertar(conn, df, tabla):
    total = len(df)
    print(f"[INFO] Insertando {total:,} filas en {SCHEMA}.{tabla}...")
    if total:
        df.to_sql(tabla, con=conn, schema=SCHEMA, if_exists='append', index=False, chunksize=CHUNK_SIZE)
    print(f"[OK] {SCHEMA}.{tabla}: {total:,} filas.")


In [ ]:
# =============================================================================
# TRANSFORMACION
# -----------------------------------------------------------------------------
# preparar_fechas: bronze crudo -> filas con Fecha (lunes). Es lo minimo para
#   saber a que semanas pertenece cada archivo.
# transformar: filas con Fecha -> (silver, outliers). Por fila y por
#   EAN_Final + Fecha, nunca cruza semanas.
# =============================================================================

def preparar_fechas(df_raw):
    # 2. COMPATIBILIDAD DE HOJA
    if '_hoja_origen' not in df_raw.columns:
        df_raw['_hoja_origen'] = 'Base Reporte'
    df_raw['_hoja_origen'] = df_raw['_hoja_origen'].fillna('Base Reporte')

    # 3. SEPARAR POR FORMATO
    # Formato nuevo: hoja 'Planilla nueva' - tiene EAN, Dinamica, Precio OFERTA.
    # Formato viejo: demas hojas - sin EAN ni dinamica.
    mask_nuevo = df_raw['_hoja_origen'] == 'Planilla nueva'
    df_viejo = df_raw[~mask_nuevo].copy()
    df_nuevo = df_raw[mask_nuevo].copy()
    print(f"   Formato viejo: {len(df_viejo):,} filas | formato nuevo: {len(df_nuevo):,} filas")

    # 4. NORMALIZAR COLUMNAS Y CONSTRUIR FECHA
    frames = []
    if len(df_viejo) > 0:
        df_viejo = normalizar_columnas_formato_viejo(df_viejo)
        df_viejo = construir_fecha(df_viejo)
        frames.append(df_viejo)
    if len(df_nuevo) > 0:
        df_nuevo = normalizar_columnas_formato_nuevo(df_nuevo)
        df_nuevo = construir_fecha(df_nuevo)
        frames.append(df_nuevo)
    if not frames:
        return df_raw.head(0).assign(Fecha=pd.NaT)

    # 5. UNIFICAR Y NORMALIZAR FECHA A LUNES
    df = pd.concat(frames, ignore_index=True)
    df = normalizar_fecha_lunes(df)

    sin_fecha = df['Fecha'].isna()
    if sin_fecha.any():
        detalle = df.loc[sin_fecha, '_archivo_origen'].value_counts().head(10)
        print(f"[WARN] {int(sin_fecha.sum()):,} filas sin fecha valida (Dia/Mes/anio no parseables) "
              f"se descartan. Por archivo (top 10):")
        print(detalle.to_string())
        df = df[~sin_fecha].copy()
    return df


def leer_maestro_eans():
    print("[INFO] Cargando maestro de EANs desde MDS_PRODU...")
    df_eans = pd.read_sql("""
        SELECT
            EAN, Reemplazo,
            Familia, Empresa, Marca, Segmento,
            [Cód SKU Marfrig]       AS Cod_SKU,
            [Descripción SKU Nuevo] AS Descripcion_SKU,
            [Marca Propia_Name]     AS Marca_Propia
        FROM [mdm].[ICO_EANs_mercado]
    """, engine_mds)
    return df_eans


def transformar(df, df_eans):
    # 6. JOIN CON MAESTRO DE EANs
    # Incluye resolucion de EAN canonico, dimensiones de producto y Marca Propia.
    df_eans = df_eans.copy()
    df_eans['EAN'] = df_eans['EAN'].astype(str)
    df_eans = df_eans.drop_duplicates(subset='EAN', keep='first')

    df['EAN'] = df['EAN'].astype(str)

    df = pd.merge(df, df_eans, on='EAN', how='left', suffixes=('', '_master'))

    # fillna('') ANTES de astype(str): la regla no puede depender de que
    # astype(str) convierta un nulo en el texto 'nan'. Con el dtype de texto
    # nuevo de pandas (por defecto desde la version 3, u opcion
    # future.infer_string) el nulo sigue nulo, el filtro != '' no lo detecta
    # y todo EAN sin reemplazo quedaba con EAN_Final nulo (detectado el
    # 17/11/2026: ~94% de las filas sin EAN en silver y gold).
    df['Reemplazo'] = (
        df['Reemplazo'].fillna('').astype(str).str.strip()
        .replace({'nan': '', 'None': '', '<NA>': ''})
    )
    df['EAN_Final'] = df['Reemplazo'].where(df['Reemplazo'] != '', df['EAN'])

    for col in ['Familia', 'Empresa', 'Marca', 'Segmento', 'Cod_SKU', 'Marca_Propia']:
        col_master = f'{col}_master'
        if col_master in df.columns:
            df[col] = df[col_master].fillna(df.get(col, pd.NA))
            df = df.drop(columns=[col_master])

    df_desc = (
        df_eans[['EAN', 'Descripcion_SKU']]
        .drop_duplicates(subset='EAN', keep='first')
        .rename(columns={'EAN': 'EAN_Final', 'Descripcion_SKU': '_desc_final'})
    )
    df = pd.merge(df, df_desc, on='EAN_Final', how='left')

    desc_canon = df['_desc_final']
    if 'Descripcion_SKU_master' in df.columns:
        desc_canon = desc_canon.fillna(df['Descripcion_SKU_master'])
    df['Descripcion_SKU'] = desc_canon.fillna(df['Descripcion_SKU'])
    df = df.drop(columns=['_desc_final'] + [c for c in ['Descripcion_SKU_master'] if c in df.columns])

    # 7. CALCULAR PRECIO_DINAMIZADO
    print("[INFO] Calculando Precio_Dinamizado...")
    df = calcular_precio_efectivo(df)

    # 8. CODIGO PDV: renombrar campo y agregar prefijo de fuente
    df['Codigo_PDV'] = 'SNP_' + df['Cod_Boca'].astype(str)

    # 8.b CADENA derivada del prefijo de Descripcion_Boca
    # "Disco - Mar del Plata" -> "Disco".
    df['Cadena'] = (
        df['Descripcion_Boca'].astype(str)
        .str.split(' - ', n=1).str[0]
        .str.replace(r'\s*%\s*$', '', regex=True)  # 'Dia %' -> 'Dia'
        .str.strip()
        .replace({'': None, 'None': None, 'nan': None})
    )

    # 9. DETECCION DE OUTLIERS (sobre Precio_BASE, banda minima 0.25-2.5xp50)
    print("[INFO] Detectando outliers sobre Precio_BASE "
          "(p50+/-8xIQR con piso p50x[0.25-2.5], criterio EAN/Fecha)...")
    df = detectar_outliers(df)

    n_outliers = int(df['_es_outlier'].sum())
    n_validos  = int((~df['_es_outlier']).sum())
    pct_outliers = n_outliers / len(df) * 100 if len(df) > 0 else 0
    print(f"   Registros validos:   {n_validos:,}")
    print(f"   Outliers detectados: {n_outliers:,} ({pct_outliers:.2f}%)")

    if n_outliers > 0:
        breakdown = (
            df[df['_es_outlier']]
            .groupby('_motivo_outlier')
            .size()
            .reset_index(name='cantidad')
        )
        print("\n   Detalle por motivo:")
        print(breakdown.to_string(index=False))

    # 10. SEPARAR VALIDOS Y OUTLIERS
    cols_silver_presentes  = [c for c in COLUMNAS_SILVER   if c in df.columns]
    cols_outlier_presentes = [c for c in COLUMNAS_OUTLIERS if c in df.columns]

    cols_silver_faltantes  = [c for c in COLUMNAS_SILVER   if c not in df.columns]
    cols_outlier_faltantes = [c for c in COLUMNAS_OUTLIERS if c not in df.columns]
    if cols_silver_faltantes:
        print(f"[WARN] Columnas esperadas no encontradas en COLUMNAS_SILVER: {cols_silver_faltantes}")
    if cols_outlier_faltantes:
        print(f"[WARN] Columnas esperadas no encontradas en COLUMNAS_OUTLIERS: {cols_outlier_faltantes}")

    df_silver   = df[~df['_es_outlier']][cols_silver_presentes].copy()
    df_outliers = df[ df['_es_outlier']][cols_outlier_presentes].copy()
    return df_silver, df_outliers


In [ ]:
# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def run_silver_snp_pipeline():
    print(f"--- INICIO PROCESO SILVER SNP: {datetime.datetime.now()} ---\n")
    print(f"[INFO] pandas {pd.__version__} | dtype de texto por defecto: "
          f"{pd.Series(['a']).dtype}")

    with engine.begin() as conn:
        asegurar_tablas_control(conn)

    # 1. MAESTRO + DECISION FULL / INCREMENTAL
    df_eans   = leer_maestro_eans()
    fp_actual = fingerprint_maestros(df_eans)

    with engine.connect() as conn:
        fp_previo     = leer_control(conn, CLAVE_MAESTROS)
        archivos_bz   = leer_archivos_bronze(conn)
        archivos_sv   = leer_archivos_silver(conn)
        silver_existe = tabla_existe(conn, TABLA_SILVER) and tabla_existe(conn, TABLA_OUTLIERS)

    motivo_full = None
    if FULL_REBUILD_ENV:
        motivo_full = 'HORUS_FULL_REBUILD=1'
    elif not silver_existe:
        motivo_full = 'tablas silver inexistentes'
    elif not archivos_sv:
        motivo_full = 'sin historial de silver (primera corrida incremental)'
    elif fp_previo != fp_actual:
        motivo_full = 'cambio en maestro de EANs MDS o en VERSION_LOGICA'
    full = motivo_full is not None

    print(f"[INFO] Bronze: {len(archivos_bz)} archivos registrados | silver: {len(archivos_sv)} procesados.")

    if full:
        print(f"[INFO] Modo: FULL REBUILD ({motivo_full}).")
        if DRY_RUN:
            print("\n[DRY-RUN] HORUS_DRY_RUN=1: no se escribe nada.")
            return
        print(f"\n[INFO] Leyendo {SCHEMA}.{TABLA_BRONZE} completo...")
        df_raw = leer_bronze(None)
        print(f"   {len(df_raw):,} filas leidas.")
        df = preparar_fechas(df_raw)
        del df_raw
        fechas = sorted(df['Fecha'].dt.date.unique())
        archivos_registrar = sorted(df['_archivo_origen'].dropna().astype(str).unique())
        eliminados = []
    else:
        cambiados, eliminados, motivos = determinar_pendientes(archivos_bz, archivos_sv)
        print(f"[INFO] Modo: INCREMENTAL. {len(cambiados)} archivos nuevos/recargados, "
              f"{len(eliminados)} eliminados en bronze.")
        for n in cambiados:
            print(f"   - {n}: {motivos[n]}")
        for n in eliminados:
            print(f"   - {n}: ya no esta en bronze, sus semanas se recalculan sin el")
        if not cambiados and not eliminados:
            print("\n[OK] Nada pendiente. Silver ya esta al dia.")
            print(f"\n--- PROCESO SILVER SNP FINALIZADO: {datetime.datetime.now()} ---")
            return

        # Lunes afectados: los de las filas nuevas de los archivos cambiados, mas
        # los que esos archivos (o los eliminados) tocaban en su version anterior.
        print(f"\n[INFO] Leyendo filas de {len(cambiados)} archivos pendientes...")
        df_pend = preparar_fechas(leer_bronze(cambiados)) if cambiados else None
        fechas = set(df_pend['Fecha'].dt.date.unique()) if df_pend is not None else set()
        for n in cambiados + eliminados:
            if n in archivos_sv:
                fechas |= archivos_sv[n][1]
        fechas = sorted(fechas)

        # Expansion: todo archivo ya procesado que aporta a esos lunes se relee.
        extra = sorted(n for n, (_, sem) in archivos_sv.items()
                       if n not in cambiados and n not in eliminados and sem & set(fechas))
        print(f"[INFO] {len(fechas)} lunes a rehacer; {len(extra)} archivos ya procesados "
              f"aportan a esos lunes y se releen.")
        for f in fechas:
            print(f"   - lunes {f}")

        if DRY_RUN:
            print("\n[DRY-RUN] HORUS_DRY_RUN=1: no se escribe nada.")
            return

        frames = [d for d in [df_pend, preparar_fechas(leer_bronze(extra)) if extra else None] if d is not None]
        df = pd.concat(frames, ignore_index=True) if frames else None
        if df is None or len(df) == 0:
            df = preparar_fechas(leer_bronze([]))
        df = df[df['Fecha'].dt.date.isin(set(fechas))].copy()
        archivos_registrar = sorted(set(cambiados) | set(extra))

    # 2. TRANSFORMAR
    print(f"\n[INFO] Transformando {len(df):,} filas...")
    if len(df):
        df_silver, df_outliers = transformar(df, df_eans)
    else:
        df_silver, df_outliers = df.head(0), df.head(0)
    semanas_por_archivo = (
        df.groupby('_archivo_origen')['Fecha'].apply(lambda s: set(s.dt.date.unique())).to_dict()
        if len(df) else {}
    )
    del df

    # 3. ESCRIBIR (una sola transaccion: silver + outliers + control)
    print("\n[INFO] Escribiendo en base de datos...")
    with engine.begin() as conn:
        if full:
            crear_tabla_desde_df(conn, df_silver,   TABLA_SILVER)
            crear_tabla_desde_df(conn, df_outliers, TABLA_OUTLIERS)
            conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_ARCHIVOS}]"))
            conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_SEMANAS}]"))
        else:
            borrar_valores(conn, TABLA_SILVER,   'Fecha', _a_datetime(fechas))
            borrar_valores(conn, TABLA_OUTLIERS, 'Fecha', _a_datetime(fechas))
            borrar_valores(conn, TABLA_ARCHIVOS, 'nombre_archivo', list(set(archivos_registrar) | set(eliminados)))
            borrar_valores(conn, TABLA_SEMANAS,  'Fecha', list(fechas))

        insertar(conn, df_silver,   TABLA_SILVER)
        insertar(conn, df_outliers, TABLA_OUTLIERS)

        if archivos_registrar:
            conn.execute(text(f"""
                INSERT INTO [{SCHEMA}].[{TABLA_ARCHIVOS}] (nombre_archivo, fecha_proceso_bronze, semanas)
                VALUES (:n, :fp, :s)
            """), [{'n': n, 'fp': archivos_bz.get(n),
                    's': _fechas_a_texto(semanas_por_archivo.get(n, set()))} for n in archivos_registrar])

        conteo_v = df_silver.groupby(df_silver['Fecha'].dt.date).size() if len(df_silver) else pd.Series(dtype=int)
        conteo_o = df_outliers.groupby(df_outliers['Fecha'].dt.date).size() if len(df_outliers) else pd.Series(dtype=int)
        fechas_res = sorted(set(conteo_v.index) | set(conteo_o.index))
        if fechas_res:
            conn.execute(text(f"""
                INSERT INTO [{SCHEMA}].[{TABLA_SEMANAS}] (Fecha, filas, filas_outliers)
                VALUES (:f, :v, :o)
            """), [{'f': f, 'v': int(conteo_v.get(f, 0)), 'o': int(conteo_o.get(f, 0))} for f in fechas_res])

        escribir_control(conn, CLAVE_MAESTROS, fp_actual)

    print(f"[OK] {len(fechas_res)} lunes escritos en silver"
          + ("" if full else f" (de {len(fechas)} afectados; los sin filas quedaron eliminados)") + ".")
    print(f"\n--- PROCESO SILVER SNP FINALIZADO: {datetime.datetime.now()} ---")


In [ ]:
run_silver_snp_pipeline()
